# Claude Code Workflow Lab: From Bug Report to Verified Fix

A runnable Jupyter notebook for practicing a safe, reviewable coding-assistant workflow. The lab creates a tiny Python project with a failing test, asks you to diagnose and fix it with Claude Code, and verifies the result.

**System design view:** This notebook follows a compact design-document structure: problem, requirements, components, workflow, verification, and operational risks.


## 1. Problem statement

We have a `divide(a, b)` function. The requirement says division by zero must raise a clear `ValueError`, but the current implementation leaks Python's default `ZeroDivisionError`. Your task is to use Claude Code to investigate, propose a minimal fix, implement it, and verify it.

### Learning outcomes

- Explain the inspect → plan → act → verify cycle.
- Give Claude a bounded task with a clear test requirement.
- Inspect changes and distinguish an explanation from evidence.
- Use a failing test to reproduce a bug and passing tests to verify a fix.


## 2. Requirements and boundaries

**Functional requirement:** `divide(a, b)` returns the quotient when `b` is nonzero and raises `ValueError` when `b` is zero.

**Constraints:** Keep the function signature. Make the smallest change that meets the requirement. Do not change unrelated files.

**Verification:** Run the focused unit test and inspect the changed file.

**Safety boundary:** This lab uses a disposable folder and contains no secrets. Review commands and edits before approving them.


## 3. System design overview

The model reasons over the task and context available in the session. Claude Code provides tools to inspect and modify files and run commands. The developer sets scope and reviews the result. The tests provide repeatable evidence about the stated requirement.

```mermaid
flowchart LR
  U[Developer task] --> A[Claude Code session]
  A --> T[File and terminal tools]
  T --> P[Practice project]
  P --> X[Tests and output]
  X --> A
  A --> R[Developer review]
  R --> U
```

| Component | Responsibility |
|---|---|
| Developer | Defines the task, boundaries, and acceptance criteria |
| Claude Code | Uses available tools to inspect files, edit code, and run commands |
| Practice project | Contains the function and unit tests |
| Test runner | Checks behavior and reports pass or fail |
| Human review | Checks the diff, scope, and whether evidence is adequate |


## 4. Prerequisites

- Python 3.9 or later and Jupyter Notebook or JupyterLab.
- Claude Code installed and authenticated if you want the agent-assisted steps. Follow Anthropic's current setup guide for installation and account requirements.
- A terminal and basic familiarity with opening a project folder.

Without Claude Code, you can still run the tests and use the instructor solution later.


In [ ]:
from pathlib import Path
import sys
import subprocess
import textwrap

PROJECT = Path.cwd() / "claude_code_practice"
PROJECT.mkdir(exist_ok=True)

buggy_code = '''
def divide(a, b):
    return a / b
'''
test_code = '''
import unittest
from calculator import divide

class TestDivide(unittest.TestCase):
    def test_divides_numbers(self):
        self.assertEqual(divide(12, 3), 4)

    def test_division_by_zero_has_clear_error(self):
        with self.assertRaises(ValueError):
            divide(12, 0)

if __name__ == '__main__':
    unittest.main()
'''

(PROJECT / "calculator.py").write_text(textwrap.dedent(buggy_code).lstrip(), encoding="utf-8")
(PROJECT / "test_calculator.py").write_text(textwrap.dedent(test_code).lstrip(), encoding="utf-8")
print(f"Practice project created at: {PROJECT}")
print("Files:", ", ".join(p.name for p in sorted(PROJECT.iterdir())))


## 5. Reproduce the bug: establish the baseline

Run the tests before asking Claude to edit anything. One test should fail because the implementation raises the wrong exception. This is the **red** stage: the test demonstrates an unmet requirement.


In [ ]:
baseline = subprocess.run(
    [sys.executable, "-m", "unittest", "-v"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(baseline.stdout)
print(baseline.stderr)
assert baseline.returncode != 0, "The starter project should have one failing test."
print("Expected result: baseline is failing. Diagnose the cause before editing.")


## 6. Inspect and plan with Claude Code

Open a terminal in the `claude_code_practice` folder. Start Claude Code there, then give it this diagnosis-only prompt. Do not ask it to edit yet.

```text
Inspect calculator.py and test_calculator.py. Explain why the zero-division test fails. Do not edit any files yet. Refer to the relevant code and suggest the smallest fix.
```

Compare its explanation with the test output. Can you point to the exact code path that produces the observed exception? If not, ask it to inspect the relevant code again.

### Learner notes

Write your diagnosis before viewing the solution:

- Expected exception: `ValueError`
- Actual exception: `ZeroDivisionError`
- Likely code location: `divide` in `calculator.py`


## 7. Implement a bounded fix

Now ask Claude Code to make the change. This prompt includes the goal, constraints, verification, and reporting expectations.

```text
Fix the division-by-zero behavior so it matches the existing test. Keep the divide(a, b) function signature. Make the smallest reasonable change and do not modify unrelated files. Run the unit tests and report the exact result.
```

Review its proposed actions. Once the edit is complete, return here and run the verification cell.


In [ ]:
print((PROJECT / "calculator.py").read_text(encoding="utf-8"))
print("--- test run ---")
result = subprocess.run(
    [sys.executable, "-m", "unittest", "-v"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Tests are still failing. Inspect the output and revise the implementation."
print("Green: all tests pass.")


## 8. Verify and review

Passing tests are useful evidence, but they do not prove the whole project is correct. Review the changed file and ask whether the implementation meets the requirement with minimal scope.

Inspect `calculator.py` directly. If you initialize Git, review `git diff` as well.

- [ ] The zero case raises `ValueError`.
- [ ] Nonzero division still returns the quotient.
- [ ] The function signature is unchanged.
- [ ] The test suite passes, and I ran it myself.
- [ ] The diff contains only relevant changes.
- [ ] The final summary matches the actual files and test output.


## 9. Instructor solution (run only after your attempt)

If you are completing this without Claude Code, or want to compare with a minimal solution, run this cell. It writes the expected implementation.


In [ ]:
solution = '''
def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b
'''
(PROJECT / "calculator.py").write_text(textwrap.dedent(solution).lstrip(), encoding="utf-8")
print((PROJECT / "calculator.py").read_text(encoding="utf-8"))


In [ ]:
final_run = subprocess.run(
    [sys.executable, "-m", "unittest", "-v"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(final_run.stdout)
print(final_run.stderr)
assert final_run.returncode == 0
print("Verified: the final implementation passes the provided tests.")


## 10. Extension challenge

Add a test for decimal values, such as `divide(7, 2) == 3.5`. Ask Claude Code to make any needed change while keeping the existing tests passing.

Then decide which of these cases belong in the function contract:

- What should happen if `a` or `b` is a string?
- Should booleans be accepted as numbers in Python?
- Should the function accept integers, floats, or both?

State a requirement and add a test before asking for new behavior.


## 11. Debrief

1. What did the test tell you that a code-only explanation could not?
2. Which part of the prompt helped keep the change small?
3. What evidence did you use to accept the fix?
4. What would you do differently if the test suite were very large?
5. Which action would you want Claude Code to ask permission for in a real repository?

### Key takeaway

Claude Code can participate directly in a coding workflow, but reliable work still depends on clear requirements, bounded actions, and independent verification.
